# 03 - Huấn luyện mô hình (Training)### Đề tài: Phân loại nấm ăn được hay có độcNotebook này huấn luyện **1 baseline + 4 mô hình** thuộc 4 họ khác nhau, cùng dùng chung 1 cách chia train/test(từ preprocess.py), cùng bộ tiền xử lý One-Hot (build_preprocessor), cùng phương pháp tinh chỉnh tham số(GridSearchCV, 5-fold Stratified Cross-Validation, scoring = F1).**Cách chạy trên Google Colab:**1. Mở notebook này trên Colab.2. Ở thanh bên trái, chọn tab Files, bấm Upload, tải lên 3 file: dataset.zip, preprocess.py, train.py   (lấy từ ai-models/data/ và ai-models/src/ trong repo).3. Runtime -> Restart session and run all.

In [ ]:
!pip -q install scikit-learn pandas joblib

In [ ]:
import syssys.path.append(".")from preprocess import load_raw_data, clean_data, get_feature_columns, encode_target, splitfrom train import train_all, get_model_specsDATA_ZIP = "dataset.zip"   # đổi đường dẫn nếu cầndf_raw = load_raw_data(DATA_ZIP)df_clean = clean_data(df_raw)feature_cols = get_feature_columns(df_clean)y, label_mapping = encode_target(df_clean)X_train, X_test, y_train, y_test = split(df_clean, feature_cols, y)print("Train:", X_train.shape, "| Test:", X_test.shape)print("Label mapping:", label_mapping)

## Danh sách model & lý do chọn| Model | Họ thuật toán | Vì sao chọn ||---|---|---|| Baseline (Dummy) | So sánh mốc | Luôn dự đoán lớp đông hơn (edible) — dùng để biết "thế nào là tốt hơn ngẫu nhiên" || Logistic Regression | Tuyến tính | Đơn giản, dễ giải thích hệ số, huấn luyện rất nhanh || KNN | Dựa trên khoảng cách | Không giả định phân phối dữ liệu, học theo "hàng xóm gần nhất" trong không gian one-hot || Random Forest | Tập thể (bagging, cây quyết định) | Bền với nhiễu, tự động chọn đặc trưng quan trọng, thường mạnh trên dữ liệu categorical || SVM | Kernel | Tìm biên phân tách tối ưu (margin lớn nhất), thử cả kernel tuyến tính và RBF |Tất cả đều được tinh chỉnh siêu tham số bằng **GridSearchCV** (5-fold Stratified CV) **chỉ trên tập train**;tập test được giữ nguyên, chỉ dùng ở bước 4 (Đánh giá).

In [ ]:
results = train_all(X_train, y_train, feature_cols, out_dir="../models/candidates")results

## Giải thích tham số đã tinh chỉnh cho từng model- **Logistic Regression**: dò `C` (nghịch đảo độ mạnh regularization) trong [0.01, 0.1, 1, 10, 100].  `C` càng nhỏ → phạt hệ số càng mạnh (tránh overfit); `C` càng lớn → mô hình càng tự do khớp dữ liệu train.- **KNN**: dò `n_neighbors` trong [3,5,7,9,11,15] và `weights` (uniform/distance).  `n_neighbors` nhỏ → nhạy nhiễu (dễ overfit); lớn → mượt hơn nhưng có thể bỏ lỡ ranh giới phức tạp.- **Random Forest**: dò `n_estimators` (số cây, [100,200]) và `max_depth` ([None,10,20]).  Nhiều cây hơn thường ổn định hơn nhưng tốn thời gian huấn luyện hơn; giới hạn `max_depth` giúp tránh overfit.- **SVM**: dò `C` ([0.1,1,10]) và `kernel` (linear/rbf).  Kernel tuyến tính phù hợp khi dữ liệu (sau one-hot) đã gần như phân tách tuyến tính được; RBF linh hoạt hơn  nhưng tốn thời gian huấn luyện hơn nhiều (thấy rõ ở thời gian train đo được bên dưới).

In [ ]:
import pandas as pdsummary = pd.DataFrame(results).Tsummary.index.name = "model"summary[["best_params","cv_score","train_time_sec","predict_time_ms","file_size_kb"]]

## Nhận xét nhanh (sẽ phân tích sâu hơn ở Bước 4 — Đánh giá trên tập TEST)- Baseline (Dummy) chỉ đạt hiệu năng ở mức "đoán lớp đông hơn" — dùng làm mốc so sánh, KHÔNG dùng metric F1 cho baseline  vì luôn dự đoán 1 lớp duy nhất (Precision/Recall của lớp thiểu số sẽ bằng 0).- 4 model còn lại đều đạt điểm F1 rất cao trên cross-validation (tập train) — phù hợp với đặc điểm đã thấy ở EDA:  một vài thuộc tính như `odor`, `spore-print-color` gần như tách biệt hoàn toàn 2 lớp.- SVM (RBF/linear) có thời gian huấn luyện lâu nhất trong 4 model — cần cân nhắc khi chọn model cuối cùng ở Bước 4  nếu ưu tiên tốc độ triển khai/dự đoán thời gian thực.- Kết quả CV trên tập train chỉ mang tính tham khảo chọn tham số — **quyết định model cuối cùng phải dựa trên  tập TEST (chưa từng thấy)**, thực hiện ở notebook `04_evaluate.ipynb`.